# Code Generator

Comments and docstrings

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important - Pause Endpoints when not in use</h1>
            <span style="color:#900;">
            If you do decide to use HuggingFace endpoints for this project, you should stop or pause the endpoints when you are done to avoid accruing unnecessary running cost. The costs are very low as long as you only run the endpoint when you're using it. Navigate to the HuggingFace endpoint UI <a href="https://ui.endpoints.huggingface.co/">here,</a> open your endpoint, and click Pause to put it on pause so you no longer pay for it.  
Many thanks to student John L. for raising this.
<br/><br/>
In week 8 we will use Modal instead of HuggingFace endpoints; with Modal you only pay for the time that you use it and you should get free credits.
            </span>
        </td>
    </tr>
</table>

# Imports

In [1]:
# imports

import os
import re
import io
import sys
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display, update_display
import gradio as gr
import subprocess
import google.generativeai as genai
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import transformers
import torch
from huggingface_hub import InferenceClient, login
from pathlib import Path
# imports
import re
import os
import sys
import textwrap
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import gradio as gr
from pathlib import Path
import subprocess
from IPython.display import Markdown
#import ollama
#!ollama pull llama3.2:1b

# Model Initialization

In [2]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['DEEPSEEK_API_KEY'] = os.getenv('DEEPSEEK_API_KEY', 'your-key-if-not-using-env')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')
login(os.environ['HF_TOKEN'], add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# initialize

openai = OpenAI()
claude = anthropic.Anthropic()
google.generativeai.configure()
OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"
GEMINI_MODEL = "gemini-2.0-flash"
LLAMA_MODEL = "codellama/CodeLlama-7b-hf"
LLAMA_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

#LLAMA_MODEL = "llama3.2:1b"
CODE_LLAMA_URL = "https://s9u5lj7g824qidsh.eu-west-1.aws.endpoints.huggingface.cloud"

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
OLLAMA_MODEL = "llama3.2"

# Code Execution

In [4]:
def extract_code(text):
    """
    Extracts Python code from a given string of text.

    This function searches the input text for a code block that is enclosed
    within ```python and ```. If such a block is found, it strips out the
    markers and returns the code inside. If no such block is found, it 
    returns an empty string and prints a notification.

    Args:
        text (str): The input text containing the code block to extract.

    Returns:
        str: The extracted Python code.
    """
    # Regular expression to find text between ```python and ```
    match = re.search(r"```python(.*?)```", text, re.DOTALL)

    if match:
        # Extract the match and strip surrounding whitespace
        code = match.group(0).strip()
    else:
        # No matching code block found
        code = ""
        print("No matching substring found.")

    # Remove the markdown markers from the matched code block
    return code.replace("```python\n", "").replace("```", "")

In [5]:
def execute_coverage_report(python_interpreter=sys.executable):
    """
    Executes pytest with coverage and returns the output.

    Args:
        python_interpreter (str, optional): Path to the Python interpreter.
            Defaults to sys.executable.

    Returns:
        str: The standard output from the coverage run.

    Raises:
        EnvironmentError: If the specified Python interpreter is not found.
    """
    if not python_interpreter:
        raise EnvironmentError("Python interpreter not found in the specified virtual environment.")
    
    command = ["coverage", "run", "-m", "pytest", "--color=no"] # Command to run coverage with pytest

    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True) # Execute the command
        print("Tests ran successfully!")
        print(result.stdout) # Print the standard output
        return result.stdout # Return the standard output
    except subprocess.CalledProcessError as e:
        print("Some tests failed!") # Notify that tests failed
        print("Output:\n", e.stdout) # Print standard output
        print("Errors:\n", e.stderr) # Print standard error
        # Extracting failed test information
        return e.stdout # Return standard output even on failure

In [6]:
def save_unit_tests(code):
    """
    Save unit tests extracted from the given code to separate files.

    This function searches for a function definition in the code,
    extracts the function name, and saves the unit tests to a file
    named after the function.

    Args:
        code (str): The code containing unit tests to be saved.

    Returns:
        None
    """
    # Search for a function definition in the code
    match = re.search(r"def\s+(\w+)\(", code, re.DOTALL)

    if match:
        # Extract the function name and remove extra spaces
        function_name = match.group(1).strip()
    else:
        # If no function is found, set an empty string and print a message
        function_name = ""
        print("No matching substring found.")

    # Define the path for saving test files
    test_code_path = Path("tests")

    # Save the extracted code to a new file named after the function
    (test_code_path / f"test_{function_name}.py").write_text(extract_code(code))

    # Remove the original test_code.py file
    Path("tests", "test_code.py").unlink()

In [7]:
def execute_tests_in_venv(code_to_test, tests, python_interpreter=sys.executable):
    """
    Execute the given Python code string and tests within the specified virtual environment.
    
    Args:
    - code_to_test: str, the Python code to be tested.
    - tests: str, the test code to be executed.
    - python_interpreter: str, path to the Python interpreter (default: sys.executable).
    
    Returns:
    - str: The output of the test execution.
    
    Raises:
    - EnvironmentError: If the Python interpreter is not found in the specified virtual environment.
    """
    
    if not python_interpreter:
        raise EnvironmentError("Python interpreter not found in the specified virtual environment.")

    # Prepare the test code by combining the code to test and the tests
    code_str = textwrap.dedent(code_to_test) + "\n" + extract_code(tests)
    
    # Create a directory for the test files
    test_code_path = Path("tests")
    test_code_path.mkdir(parents=True, exist_ok=True)
    
    # Write the combined code to a file
    (test_code_path / f"test_code.py").write_text(code_str)
    
    # Prepare the command to execute pytest
    command = ["pytest", "--color=no", str(test_code_path)]

    try:
        # Run the tests using subprocess
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        print("Tests ran successfully!")
        print(result.stderr)
        return result.stdout
    except subprocess.CalledProcessError as e:
        print("Some tests failed!")
        print("Output:\n", e.stdout)
        print("Errors:\n", e.stderr)
        
        # Extract and print information about failed tests
        failed_tests = []
        for line in e.stdout.splitlines():
            if "FAILED" in line and "::" in line:
                failed_tests.append(line.strip())
        if failed_tests:
            print("Failed Tests:")
            for test in failed_tests:
                print(test)
    
        return e.stdout

# Prompts and Calls to the Models

In [8]:
system_message = "You are a helpful code assistant which helps developers to write unit test cases for their code using pytest and coverage."
system_message += "Do not respond with greetings, or any such extra output."
system_message += "You are allowed to insert ```python and ``` at the beginning and at the end, respectively."
system_message += "Do not add any note or any explanation about your test units at the end."

In [9]:
def user_prompt_for(code, insert_function):

    if not insert_function:
        user_prompt = """Test include:

        - Valid inputs with expected results.
        - Inputs that test the boundaries or limits of the function's behavior.
        - Invalid inputs or scenarios where the function is expected to raise exceptions.

        Structure:

        - Begin with all necessary imports. 
        - Do not create custom imports. 
        - Do not insert in the response the function for the tests.
        - Ensure proper error handling for tests that expect exceptions.
        - Clearly name the test functions to indicate their purpose (e.g., test_function_name).

        Example Structure:

        - Use pytest.raises to validate exceptions.
        - Use assertions to verify correct outputs for successful and edge cases.

        Documentation:

        - Add docstrings explaining what each test verifies."""
    
    else:
        user_prompt = """Test include:

        - Valid inputs with expected results.
        - Inputs that test the boundaries or limits of the function's behavior.
        - Invalid inputs or scenarios where the function is expected to raise exceptions.

        Structure:

        - Begin with all necessary imports. 
        - Do not create custom imports. 
        - Insert in the response the function for the tests.
        - Ensure proper error handling for tests that expect exceptions.
        - Clearly name the test functions to indicate their purpose (e.g., test_function_name).

        Example Structure:

        - Use pytest.raises to validate exceptions.
        - Use assertions to verify correct outputs for successful and edge cases.

        Documentation:

        - Add docstrings explaining what each test verifies."""
    
    user_prompt += code

    return user_prompt

In [10]:
def system_user_prompt_for(code, insert_function):

    system_prompt = system_message

    if not insert_function:
        user_prompt = """Test include:

        - Valid inputs with expected results.
        - Inputs that test the boundaries or limits of the function's behavior.
        - Invalid inputs or scenarios where the function is expected to raise exceptions.

        Structure:

        - Begin with all necessary imports. 
        - Do not create custom imports. 
        - Do not insert in the response the function for the tests.
        - Ensure proper error handling for tests that expect exceptions.
        - Clearly name the test functions to indicate their purpose (e.g., test_function_name).

        Example Structure:

        - Use pytest.raises to validate exceptions.
        - Use assertions to verify correct outputs for successful and edge cases.

        Documentation:

        - Add docstrings explaining what each test verifies."""
    
    else:
        user_prompt = """Test include:

        - Valid inputs with expected results.
        - Inputs that test the boundaries or limits of the function's behavior.
        - Invalid inputs or scenarios where the function is expected to raise exceptions.

        Structure:

        - Begin with all necessary imports. 
        - Do not create custom imports. 
        - Insert in the response the function for the tests.
        - Ensure proper error handling for tests that expect exceptions.
        - Clearly name the test functions to indicate their purpose (e.g., test_function_name).

        Example Structure:

        - Use pytest.raises to validate exceptions.
        - Use assertions to verify correct outputs for successful and edge cases.

        Documentation:

        - Add docstrings explaining what each test verifies."""
    
    prompt = system_prompt + "\n\n" + user_prompt + "\n\nThis is the code:\n\n" + code

    return prompt

In [11]:
def messages_for(code, insert_function):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(code, insert_function)}
    ]

In [12]:
def stream_gpt(code, insert_function):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(code, insert_function), stream=True)
    reply = ""
    for chunk in stream:        
        reply += chunk.choices[0].delta.content or ""
        yield reply
    
    return reply

In [13]:
def stream_claude(code, insert_function):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(code, insert_function)}],
    )
    reply = ""    
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply
    
    return reply

In [14]:
def stream_gemini(code, insert_function):
    model = genai.GenerativeModel(GEMINI_MODEL)
    prompt = system_user_prompt_for(code, insert_function)
    result = model.generate_content(prompt, stream=True)
    reply = ""
    for chunk in result:
        reply += chunk.text or ""
        yield reply 
    
    return reply

# Code Examples

In [15]:
function_to_test = """
    def lengthOfLongestSubstring(s):
        if not isinstance(s, str):
            raise TypeError("Input must be a string")
        max_length = 0
        substring = ""
        start_idx = 0
        while start_idx < len(s):
            string = s[start_idx:]
            for i, x in enumerate(string):
                substring += x
                if len(substring) == len(set((list(substring)))):
                    
                    if len(set((list(substring)))) > max_length:
                        
                        max_length = len(substring)

            start_idx += 1
            substring = ""
                  
                
        return max_length"""

In [16]:
test_code = """```python
import pytest

# Unit tests using pytest
def test_lengthOfLongestSubstring():
    assert lengthOfLongestSubstring("abcabcbb") == 3  # Case with repeating characters
    assert lengthOfLongestSubstring("bbbbb") == 1    # Case with all same characters
    assert lengthOfLongestSubstring("pwwkew") == 3    # Case with mixed characters
    assert lengthOfLongestSubstring("") == 0           # Empty string case
    assert lengthOfLongestSubstring("abcdef") == 6     # All unique characters
    assert lengthOfLongestSubstring("abca") == 3       # Case with pattern and repeat
    assert lengthOfLongestSubstring("dvdf") == 3       # Case with repeated characters separated
    assert lengthOfLongestSubstring("a") == 1           # Case with single character
    assert lengthOfLongestSubstring("au") == 2          # Case with unique two characters
```"""

In [17]:
def generate_unit_tests(code, insert_function=False, model="GPT"):
    if model=="GPT":
        result = stream_gpt(code, insert_function)
    elif model=="Claude":
        result = stream_claude(code, insert_function)
    elif model=="Gemini":
        result = stream_gemini(code, insert_function)
    else:
        raise ValueError("Unknown model")
    
    for stream_so_far in result:
        yield stream_so_far        

In [18]:
css = """
.without {background-color: #306998; label: black}
.with {background-color: #050; label: black}
"""

In [20]:
with gr.Blocks() as ui:
    gr.Markdown("## Write unit tests for Python code")
    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            python = gr.Textbox(label="Python code:", value=function_to_test, lines=20)            
            model = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")
            insert_fn = gr.Checkbox(label="Insert function in response", value=False)            
        with gr.Column(scale=1, min_width=300):            
            unit_tests_out = gr.TextArea(label="Unit tests", value=test_code, elem_classes=["python"])
            unit_tests = gr.Button("Generate unit tests")
            unit_tests_run = gr.Button("Run unit tests")
            coverage_run = gr.Button("Coverage report")
            save_test_run = gr.Button("Save unit tests")
    with gr.Row():
        
        python_out = gr.TextArea(label="Unit tests result", elem_classes=["python"])
        coverage_out = gr.TextArea(label="Coverage report", elem_classes=["python"])
        

    unit_tests.click(generate_unit_tests, inputs=[python, insert_fn, model], outputs=[unit_tests_out])
    unit_tests_run.click(execute_tests_in_venv, inputs=[python, unit_tests_out], outputs=[python_out])
    coverage_run.click(execute_coverage_report, outputs=[coverage_out])
    save_test_run.click(save_unit_tests, inputs=[unit_tests_out])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


Tests ran successfully!

Some tests failed!
Output:
 ============================= test session starts =============================
platform win32 -- Python 3.12.3, pytest-8.3.3, pluggy-1.5.0
rootdir: c:\Users\ssre_\Projects\LLM-Engineering\week4_leaderboards_code_generation
plugins: anyio-4.1.0, dash-2.15.0, cov-5.0.0, mock-3.14.0
collected 12 items

tests\test_code.py ......F...F.                                          [100%]

================================== FAILURES ===================================
_______________ test_lengthOfLongestSubstring_string_with_space _______________

    def test_lengthOfLongestSubstring_string_with_space():
        """Test with string including space."""
        s = "abcde fgh"
>       assert lengthOfLongestSubstring(s) == 8
E       AssertionError: assert 9 == 8
E        +  where 9 = lengthOfLongestSubstring('abcde fgh')

tests\test_code.py:58: AssertionError
______________ test_lengthOfLongestSubstring_string_with_newline ______________

    de